# L2 step 4b · The chosen 3B on the benchmark: the clean score, and the counting options' pattern (your PC, no GPU)

Task **L2**, objective **O4**. Run 5 (`l2-step4-winner-benchmark`, 23 Nov) scored the chosen 3B, option (c), on the benchmark's 10,675 optical questions. It came out ahead of our 7B on building counting (48.2% against 30.3%), road damage counting (44.7% against 33.0%) and relational reasoning (51.6% against 34.6%). Two checks come before any of that is claimed. They use only answers already saved, so no model runs.

**Check 1, the clean score (E5's rule, 26 Oct).** 186 of the 10,675 questions use an after-photo that is also in DisasterM3's training images (found by image content in E5 step 1). Every score is reported three ways:
- *all* questions;
- *clean*, without those 186 (the rule fixed on 26 Oct);
- *strict*, without all 512 flagged questions (adding before-photo copies and review matches).

This is done for the 3B in normal precision (run 5), the 3B in 4 bits (L3) and our 7B (L0 step 2, at full size).

**Check 2, the counting options' pattern.** The benchmark builds each counting question's five options around the true count. When the largest option is above 20, the middle option is the right one. A rule that never sees the photo (largest above 20 → the middle; otherwise the second smallest) scores 67.0% on building counting. The 3B (c) trained on the 7B's answers with these options shown, so it could have learned the pattern instead of counting. So the counting questions are split in two:
- *fair*: largest option 20 or less, where the middle isn't always right;
- *pattern*: largest option above 20, where it is.

For each model the check reports accuracy on each part and how often it picks the middle on the pattern part. Both counting tasks are covered (buildings and roads).

**Rule, fixed before the run (23 Nov):** the 3B's counting lead counts as more than the options' pattern only if, on the fair counting questions (both tasks together), the 3B in normal precision beats both of these by more than chance (exact McNemar test, p < 0.05):
- our 7B;
- the rule that never sees the photo.

Otherwise the report says only that it scores higher on these options.

**Expectation, fixed before the run:**
- *Clean:* every model's score on every task moves by at most 1 point, and no task changes which of the 3B and 7B is ahead. Only 1.74% of the questions are copies.
- *Pattern:* the 3B picks the middle on the pattern questions far more often than our 7B, at least 1.5 times as often. This follows from its training with the options shown: on run 4's held-out questions it picked the middle 39% of the time, against 20% for the untrained 3B.
- *Fair questions:* the 3B still beats the 7B and the blind rule. On run 4's held-out fair questions it got 56.4%, against 39.0% for the blind rule.

**Why this test, and the alternatives**
- The fair/pattern split comes from how the options are built, not from the results, so it can't be tuned to them.
- *Paired tests:* every model answers the same questions, so McNemar's exact test (on the questions only one side gets right) is the paired test, as in run 4's rule.
- *Rejected: dropping the pattern questions from the benchmark score itself.* That would change the published task, so the full score stays the headline and the fair score sits beside it.
- *Still possible:* count error against the true count (run 5 reports a median), and a counting test with options built differently. The latter would need new questions; E7's chain counts against E6's true masks.

**How to run:** on the PC, in Jupyter (cse499 environment), from `Champion_Task/`. Run all; it takes seconds. Writes `L2-step4-results/full/checks/`.

In [1]:
# ── Cell 1: The saved answers: the 3B in fp16 (run 5) and in 4 bits (L3), our 7B (L0 step 2), and E5's flags ──
import ast
import json
import math
import re
from pathlib import Path

import numpy as np
import pandas as pd

CT = Path.cwd()                                           # Champion_Task
FP16_DIR = CT / "L2-step4-results" / "full" / "L2S4"
NF4_DIR = CT / "L3-results" / "full" / "L3"
SEVEN_B = [CT / "L0-step2-results" / p / "L0S2" / "L0_step2_new_per_question.csv" for p in ("new-partA", "new-partB")]
FLAGGED = CT / "E5-results" / "step1" / "bench_questions_flagged.csv"
OUT = CT / "L2-step4-results" / "full" / "checks"
OUT.mkdir(parents=True, exist_ok=True)
TASKS = ["DTR", "BBR", "BDC", "DRE", "DSR", "ORR"]
COUNTING = ["BDC", "DRE"]
for d in (FP16_DIR, NF4_DIR):
    cfg = json.loads((d / "config.json").read_text(encoding="utf-8"))
    assert not cfg["smoke_test"] and cfg["complete"] and cfg["winner"] == "c", f"{d}: not the full run of (c)"

def letters(text):
    """The benchmark scorer's letters (L0 step 2, run 5, L3)."""
    t = str(text).upper().replace("THE ANSWER IS", "").replace("OPTION", "")
    return set(re.findall(r"[A-Z]", t))

q = pd.read_csv(FP16_DIR / "per_question.csv")
nf4 = pd.read_csv(NF4_DIR / "per_question.csv", usecols=["id", "answer", "correct"]).set_index("id")
seven = pd.concat([pd.read_csv(p) for p in SEVEN_B]).dropna(subset=["answer"]).drop_duplicates("id").set_index("id")
q["answer_4bit"], q["correct_4bit"] = q["id"].map(nf4["answer"]), q["id"].map(nf4["correct"]).astype(bool)
q["answer_7b"], q["correct_7b"] = q["id"].map(seven["answer"]), q["id"].map(seven["correct"]).fillna(False).astype(bool)
q["correct_3b"] = q["correct"].astype(bool)
assert len(q) == 10675 and q["id"].is_unique and q["id"].isin(nf4.index).all()
# 8 benchmark images can't be read: all three models have no answer for the same 8 questions, counted wrong (as in L0 step 2)
unread = q["answer"].isna()
assert unread.sum() == 8 and (unread == q["answer_4bit"].isna()).all() and (unread == q["answer_7b"].isna()).all()
flag = pd.read_csv(FLAGGED)
flag = flag[flag["sensor"] == "Optical"]
copied = set(flag.loc[flag["flag"] == "after-image copied", "id"])
flagged_any = set(flag["id"])
q["clean"], q["strict"] = ~q["id"].isin(copied), ~q["id"].isin(flagged_any)
assert q["id"].isin(copied).sum() == 186
# Our 7B's saved scores reproduce L0 step 2's published ones
REF_7B = {"DTR": 76.77, "BBR": 19.99, "BDC": 30.31, "DRE": 33.03, "DSR": 40.46, "ORR": 34.62}
for t in TASKS:
    got = round(100 * q.loc[q["task"] == t, "correct_7b"].mean(), 2)
    assert abs(got - REF_7B[t]) < 0.011, f"7B {t}: {got} against {REF_7B[t]}"
print(f"✓ {len(q):,} questions; 7B answered {int(q['answer_7b'].notna().sum()):,}; clean {int(q['clean'].sum()):,}, "
      f"strict {int(q['strict'].sum()):,}; the 7B's six published scores reproduced")

✓ 10,675 questions; 7B answered 10,667; clean 10,489, strict 10,289; the 7B's six published scores reproduced


In [2]:
# ── Cell 2: Check 1, the clean score: all, clean (without the 186 copies) and strict (without all 512 flagged) ──
MODELS = {"3B (c), fp16": "correct_3b", "3B (c), 4 bits": "correct_4bit", "our 7B": "correct_7b"}
rows = []
for name, col in MODELS.items():
    for t in TASKS + ["mean of six"]:
        row = {"model": name, "task": t}
        for subset in ("all", "clean", "strict"):
            d = q if subset == "all" else q[q[subset]]
            if t == "mean of six":
                row[subset] = round(float(np.mean([100 * d.loc[d["task"] == k, col].mean() for k in TASKS])), 2)
            else:
                row[subset] = round(100 * float(d.loc[d["task"] == t, col].mean()), 2)
                row[f"n_{subset}"] = int((d["task"] == t).sum())
        row["clean_minus_all"] = round(row["clean"] - row["all"], 2)
        rows.append(row)
clean = pd.DataFrame(rows)
wide = (clean.assign(v=lambda d: d["all"].map("{:.2f}".format) + " → " + d["clean"].map("{:.2f}".format)
                     + " (" + d["strict"].map("{:.2f}".format) + ")")
        .pivot(index="task", columns="model", values="v").reindex(TASKS + ["mean of six"]))
flips = [t for t in TASKS
         if (clean.query("model == '3B (c), fp16' and task == @t")["all"].item() > clean.query("model == 'our 7B' and task == @t")["all"].item())
         != (clean.query("model == '3B (c), fp16' and task == @t")["clean"].item() > clean.query("model == 'our 7B' and task == @t")["clean"].item())]
print(clean.to_string(index=False))
print(f"\nLargest move, all to clean: {clean['clean_minus_all'].abs().max():.2f} points; tasks where the 3B/7B order flips: {flips or 'none'}")

         model        task   all  n_all  clean  n_clean  strict  n_strict  clean_minus_all
  3B (c), fp16         DTR 62.04  353.0  60.93    343.0   60.30     335.0            -1.11
  3B (c), fp16         BBR 16.65 2006.0  16.39   1971.0   16.12    1929.0            -0.26
  3B (c), fp16         BDC 49.79 3583.0  49.62   3517.0   49.30    3452.0            -0.17
  3B (c), fp16         DRE 46.63 1868.0  46.29   1834.0   46.19    1797.0            -0.34
  3B (c), fp16         DSR 35.38 2007.0  35.09   1972.0   34.61    1930.0            -0.29
  3B (c), fp16         ORR 51.98  858.0  52.23    852.0   52.13     846.0             0.25
  3B (c), fp16 mean of six 43.74    NaN  43.43      NaN   43.11       NaN            -0.31
3B (c), 4 bits         DTR 63.17  353.0  62.10    343.0   61.49     335.0            -1.07
3B (c), 4 bits         BBR 16.55 2006.0  16.44   1971.0   16.07    1929.0            -0.11
3B (c), 4 bits         BDC 48.17 3583.0  48.00   3517.0   47.62    3452.0            -0.17

In [3]:
# ── Cell 3: Check 2, the counting options' pattern: fair (largest option ≤ 20) and pattern (> 20) questions ──
def option_values(v):
    return [float(x) for x in (ast.literal_eval(v) if isinstance(v, str) else v)]

def middle_letter(values):
    return "ABCDE"[values.index(sorted(values)[2])]

def blind_rule(values):
    """Never sees the photo (run 5's and L0's rule): largest above 20 -> the middle value, otherwise the second smallest."""
    s = sorted(values)
    return "ABCDE"[values.index(s[2] if s[4] > 20 else s[1])]

assert blind_rule([231, 323, 277, 139, 185]) == "A" and middle_letter([5, 1, 3, 2, 4]) == "C"
c = q[q["task"].isin(COUNTING)].copy()
c["values_list"] = c["values"].map(option_values)
c["part"] = np.where(c["values_list"].map(max) > 20, "pattern", "fair")
c["truth_letter"] = c["truth"].map(lambda t: "".join(sorted(letters(t))))
c["middle_right"] = [middle_letter(v) == t for v, t in zip(c["values_list"], c["truth_letter"])]
c["correct_rule"] = [blind_rule(v) == t for v, t in zip(c["values_list"], c["truth_letter"])]
assert (c["correct_rule"] == c["rule_correct"].astype(bool)).all(), "the blind rule doesn't reproduce run 5's"
for col, ans in (("3b", "answer"), ("4bit", "answer_4bit"), ("7b", "answer_7b")):
    c[f"middle_{col}"] = [letters(a) == {middle_letter(v)} for a, v in zip(c[ans].fillna(""), c["values_list"])]

def part_row(d, task, part):
    return {"task": task, "part": part, "questions": len(d), "middle_is_right_%": round(100 * d["middle_right"].mean(), 1),
            "blind_rule_%": round(100 * d["correct_rule"].mean(), 2), "3B_fp16_%": round(100 * d["correct_3b"].mean(), 2),
            "3B_4bit_%": round(100 * d["correct_4bit"].mean(), 2), "7B_%": round(100 * d["correct_7b"].mean(), 2),
            "3B_picks_middle_%": round(100 * d["middle_3b"].mean(), 1), "7B_picks_middle_%": round(100 * d["middle_7b"].mean(), 1)}

pattern = pd.DataFrame([part_row(c[(c["task"].isin(ts)) & (c["part"] == p)], name, p)
                        for name, ts in (("BDC", ["BDC"]), ("DRE", ["DRE"]), ("both", COUNTING)) for p in ("fair", "pattern")]
                       + [part_row(c[c["task"].isin(ts) & c["clean"] & (c["part"] == "fair")], f"{name}, clean", "fair")
                          for name, ts in (("both", COUNTING),)])
print(pattern.to_string(index=False))

       task    part  questions  middle_is_right_%  blind_rule_%  3B_fp16_%  3B_4bit_%  7B_%  3B_picks_middle_%  7B_picks_middle_%
        BDC    fair       2002               17.6         40.91      57.39      53.85 31.97               16.0               20.9
        BDC pattern       1581              100.0        100.00      40.16      40.99 28.21               40.2               28.2
        DRE    fair       1560               33.1         30.06      48.14      51.15 34.81               29.9               28.2
        DRE pattern        308              100.0        100.00      38.96      36.36 24.03               39.0               24.0
       both    fair       3562               24.4         36.16      53.34      52.67 33.21               22.1               24.1
       both pattern       1889              100.0        100.00      39.97      40.23 27.53               40.0               27.5
both, clean    fair       3473               24.4         36.08      53.04      52.35 32.9

In [4]:
# ── Cell 4: The rule: on the fair counting questions, the 3B against our 7B and against the blind rule ──
def mcnemar_exact(only_a, only_b):
    """Two-sided exact McNemar: the binomial test on the questions only one side gets right."""
    n = only_a + only_b
    if n == 0:
        return 1.0
    k = min(only_a, only_b)
    return min(1.0, 2 * sum(math.comb(n, i) for i in range(k + 1)) / 2 ** n)

assert abs(mcnemar_exact(0, 5) - 0.0625) < 1e-12 and mcnemar_exact(5, 5) == 1.0 and mcnemar_exact(338, 185) < 1e-4   # run 4: p < 0.0001
fair = c[c["part"] == "fair"]
tests = []
for other, col in (("our 7B", "correct_7b"), ("the blind rule", "correct_rule")):
    only_3b = int((fair["correct_3b"] & ~fair[col]).sum())
    only_other = int((~fair["correct_3b"] & fair[col]).sum())
    tests.append({"3B (c), fp16 against": other, "questions": len(fair), "3B_%": round(100 * fair["correct_3b"].mean(), 2),
                  "other_%": round(100 * fair[col].mean(), 2), "only_3B_right": only_3b, "only_other_right": only_other,
                  "p": mcnemar_exact(only_3b, only_other)})
tests = pd.DataFrame(tests)
passes = bool(((tests["3B_%"] > tests["other_%"]) & (tests["p"] < 0.05)).all())
VERDICT = ("passes: on the fair counting questions the 3B beats our 7B and the blind rule by more than chance, so its counting "
           "lead is more than the options' pattern" if passes else
           "doesn't pass: the 3B's counting lead can be claimed only as a higher score on these options")
print(tests.to_string(index=False))
print(f"\nRule: {VERDICT}")

3B (c), fp16 against  questions  3B_%  other_%  only_3B_right  only_other_right             p
              our 7B       3562 53.34    33.21            871               154 5.367154e-122
      the blind rule       3562 53.34    36.16           1083               471  1.286317e-55

Rule: passes: on the fair counting questions the 3B beats our 7B and the blind rule by more than chance, so its counting lead is more than the options' pattern


In [5]:
# ── Cell 5: Expectation against result, and the files ─────────────────────────────
def md(df):
    try:
        return df.to_markdown(index=False)
    except ImportError:              # tabulate isn't in the cse499 environment
        return "```\n" + df.to_string(index=False) + "\n```"

pat = pattern.set_index(["task", "part"])
expect = pd.DataFrame([
    ("clean: every score moves by at most 1 point", f"largest move {clean['clean_minus_all'].abs().max():.2f} points",
     bool(clean["clean_minus_all"].abs().max() <= 1.0)),
    ("clean: no task changes which of the 3B and 7B is ahead", f"flips: {', '.join(flips) or 'none'}", not flips),
    ("the 3B picks the middle on pattern questions far more often than the 7B",
     f"{pat.loc[('both', 'pattern'), '3B_picks_middle_%']}% against {pat.loc[('both', 'pattern'), '7B_picks_middle_%']}%",
     bool(pat.loc[("both", "pattern"), "3B_picks_middle_%"] >= 1.5 * pat.loc[("both", "pattern"), "7B_picks_middle_%"])),
    ("on fair questions the 3B beats the 7B and the blind rule", VERDICT.split(":")[0], passes),
], columns=["expectation (23 Nov)", "result", "met"])
print(expect.to_string(index=False))
clean.to_csv(OUT / "clean_scores.csv", index=False)
pattern.to_csv(OUT / "counting_options_pattern.csv", index=False)
tests.to_csv(OUT / "fair_counting_tests.csv", index=False)
c.drop(columns=["images", "values_list"], errors="ignore").to_csv(OUT / "counting_per_question.csv", index=False)
(OUT / "README.md").write_text("\n".join([
    "# L2 step 4b: the chosen 3B's clean score and the counting options' pattern", "",
    "From saved answers only, by `l2-step4b-clean-and-options-check.ipynb` on the PC, with the rule and expectation fixed "
    "in its first cell before the run. The 3B is option (c) in normal precision (run 5) and in 4 bits (L3). Our 7B is "
    "L0 step 2's, at full size.", "",
    "## Check 1: the clean score", "",
    "Each cell is all → clean (strict), in %. Clean drops the 186 questions whose after-photo is in DisasterM3's training "
    "images; strict drops all 512 flagged.", "",
    md(wide.reset_index().rename_axis(None, axis=1)), "",
    f"Largest move from all to clean: {clean['clean_minus_all'].abs().max():.2f} points. Tasks where the 3B/7B order "
    f"flips: {', '.join(flips) or 'none'}. Every model, task and subset: `clean_scores.csv`.", "",
    "## Check 2: the counting options' pattern", "",
    "Fair: largest option 20 or less. Pattern: largest above 20, where the middle option is the right one.", "",
    md(pattern), "",
    "## The rule (fixed before the run)", "", md(tests), "", f"**{VERDICT[0].upper() + VERDICT[1:]}.**", "",
    "## Expectation against result", "", md(expect), "",
    "## Files", "", "`clean_scores.csv`, `counting_options_pattern.csv`, `fair_counting_tests.csv`, `counting_per_question.csv`",
]) + "\n", encoding="utf-8")
print(f"✓ Wrote {OUT}")

                                                   expectation (23 Nov)                   result   met
                            clean: every score moves by at most 1 point largest move 1.11 points False
                 clean: no task changes which of the 3B and 7B is ahead              flips: none  True
the 3B picks the middle on pattern questions far more often than the 7B      40.0% against 27.5% False
               on fair questions the 3B beats the 7B and the blind rule                   passes  True


✓ Wrote d:\CSE499AB_project\Essential_task_delegation\Champion_Task\L2-step4-results\full\checks
